# Project 18 - Notebook 02: Giai Đoạn 1 - Zero-DCE & CLAHE Enhancement
Huấn luyện mô hình tự giám sát Zero-DCE và đo đạc chất lượng ảnh không tham chiếu (NIQE, BRISQUE).

In [ ]:
import sys
sys.path.append('..')
import torch
import cv2
import matplotlib.pyplot as plt
import numpy as np
from src.model_zerodce import DCENet, ZeroDCEpp, enhance_image
from src.loss_zerodce import ZeroDCELoss
from src.preprocess_dip import enhance_clahe_bilateral
from src.metrics import calculate_niqe, calculate_brisque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### 1. Khởi tạo mô hình DCE-Net và Zero-DCE++

In [ ]:
model_dce = DCENet()
model_pp = ZeroDCEpp()

p_dce = sum(p.numel() for p in model_dce.parameters())
p_pp = sum(p.numel() for p in model_pp.parameters())
print(f'DCENet Parameters:   {p_dce:,}')
print(f'Zero-DCE++ Parameters: {p_pp:,} (~1/7 dung lượng)')

### 2. So sánh trực quan: Raw Dark vs CLAHE vs Zero-DCE

In [ ]:
sample_path = list(Path('../Dataset/exdark_yolo_dark/test/images').glob('*.*'))[0]
raw_bgr = cv2.imread(str(sample_path))
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

# 1. CLAHE + Bilateral
clahe_bgr = enhance_clahe_bilateral(raw_bgr)
clahe_rgb = cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB)

# 2. Zero-DCE
model_dce.eval()
t = torch.from_numpy(raw_rgb).float().permute(2, 0, 1).unsqueeze(0) / 255.0
with torch.no_grad():
    enh_t, A = model_dce(t)
dce_rgb = (enh_t.squeeze(0).permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)

# Trực quan hóa
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(raw_rgb)
axes[0].set_title(f'Raw Dark (NIQE: {calculate_niqe(raw_bgr):.2f})')
axes[0].axis('off')

axes[1].imshow(clahe_rgb)
axes[1].set_title(f'CLAHE Baseline (NIQE: {calculate_niqe(clahe_bgr):.2f})')
axes[1].axis('off')

axes[2].imshow(dce_rgb)
axes[2].set_title(f'Zero-DCE (NIQE: {calculate_niqe(cv2.cvtColor(dce_rgb, cv2.COLOR_RGB2BGR)):.2f})')
axes[2].axis('off')

plt.tight_layout()
plt.show()